# RNN clásica (ejemplo)
Este notebook muestra cómo usar la implementación RNN integrada en la librería `fatigueset-lib`.
Se intentará: cargar el pipeline, generar ventanas, y lanzar un entrenamiento corto con `train_kfold` (si PyTorch está disponible).

In [1]:
# Asegura que la librería local `fatigueset-lib` esté en sys.path
import sys
from pathlib import Path
p = Path.cwd()
while not (p / 'fatigueset-lib').exists() and p.parent != p:
    p = p.parent
lib_path = str(p / 'fatigueset-lib')
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)
print('fatigueset-lib added to sys.path ->', lib_path)
import importlib
import fatigueset
exports = [k for k in dir(fatigueset) if k in ('train_kfold','RNNFatiga','FatigueSequenceDataset') or 'rnn' in k.lower()]
print('fatigueset exports (filtered):', exports)


fatigueset-lib added to sys.path -> c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib
fatigueset exports (filtered): ['FatigueSequenceDataset', 'RNNFatiga', 'rnn', 'train_kfold']


In [2]:
# Cargar pipeline y generar ventanas (modo seguro, parámetros pequeños)
from pathlib import Path
from fatigueset import FatigueSetPipeline

p = Path.cwd()
while not (p / 'fatigueset').exists() and p.parent != p:
    p = p.parent
DATASET_PATH = str(p / 'fatigueset')
print('Using dataset path ->', DATASET_PATH)

pipeline = FatigueSetPipeline(dataset_path=DATASET_PATH)
res = pipeline.ejecutar(verbose=False, incluir_ventanas=False, normalizar=True)
df_ml = res.get('ml')
print('ml dataframe shape:', None if df_ml is None else df_ml.shape)
df_windows = pipeline.crear_ventanas(df_ml, window_size=16, step=8)
print('windows dataframe shape:', df_windows.shape)
print(df_windows.head().to_string(index=False))


Using dataset path -> c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset
ml dataframe shape: (108, 31)
windows dataframe shape: (0, 0)
Empty DataFrame
Columns: []
Index: []


c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [3]:
# Intentar entrenamiento corto con train_kfold (se captura excepción si falta PyTorch)
from fatigueset import train_kfold
import traceback
try:
    results = train_kfold(pipeline=pipeline, window_size=2, step=1, seq_len=2, epochs=2, batch_size=16, n_splits=2, output_dir='output/rnn_test')
    print('train_kfold results:', results)
except Exception as e:
    print('train_kfold skipped or failed:', repr(e))
    traceback.print_exc()


Cargando dataset y construyendo dataframe ML...


c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


Construyendo ventanas (incluir targets)


c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4597: RuntimeWarning: invalid value encountered in add
  lerp_interpolation = add(a, diff_b_a * t, out=... if out is None else out)
c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4597: RuntimeWarning: invalid value encountered in add
  lerp_interpolation = add(a, diff_b_a * t, out=... if out is None else out)
c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4597: RuntimeWarning: invalid value encountered in add
  lerp_interpolation = add(a, diff_b_a * t, out=... if out is None else out)
c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\.venv\Lib\site-packages\pandas\core\n

Fold 1/2: train 18 samples, test 18 samples
 epoch 1: train_loss=1493.524475 val_loss=1116.156006
 epoch 2: train_loss=1483.369934 val_loss=1100.795135
Fold 2/2: train 18 samples, test 18 samples
 epoch 1: train_loss=1724.957336 val_loss=901.388458
 epoch 2: train_loss=1011.928406 val_loss=880.311508
train_kfold results: {'fold_1': {'mse_fisica': 759.3723754882812, 'mae_fisica': 22.98098373413086, 'r2_fisica': -2.1918487548828125, 'mse_mental': 2044.16259765625, 'mae_mental': 40.28409957885742, 'r2_mental': -3.8534388542175293, 'n_test': 18}, 'fold_2': {'mse_fisica': 953.1799926757812, 'mae_fisica': 25.941078186035156, 'r2_fisica': -2.315211057662964, 'mse_mental': 1453.953369140625, 'mae_mental': 31.507368087768555, 'r2_mental': -2.104194402694702, 'n_test': 18}}
